# Boundary mass

A column card gives a numeric column's minimum and maximum as two
numbers, whether the edge values are single readings or crowds.
The `boundaries` detector counts what sits at the edges. When one
value at the observed minimum or maximum repeats far more often
than the values just inside it, the column carries mass piled at
its boundary — the shape a recording cap or floor leaves, and the
shape a genuine bound of the measured quantity leaves as well. In
a column that is mostly zeros, the same comparison runs at
interior values, where a repeated exact amount can stand above
its neighbours on both sides. The detector is in the default set
and emits `formats` findings of type `boundary_mass`.

The worked example is the Adult census table from the benchmark
corpus (48,842 rows, 15 columns), profiled at the standard
budget. Its capital columns carry both shapes the check reports,
and its largest repeated value earns no statement — that silence
is part of the method, and is taken up below. A second example,
the Metro traffic table's temperature column, is the floor case.
The neighbouring scan is [Sentinel values](sentinel_values.ipynb),
which looks for a repeated value concentrated in a stretch of
rows; this check counts over the whole column and never looks at
where the rows sit, and on the Adult table the sentinel pass
reports none of the ten piles below.

In [ ]:
import polars as pl

import onceover

df = pl.read_csv(
    "../../../benchmarks/corpus/data/business/adult-census-income/adult-census-income_uci-2.csv"
)
report = onceover.profile(df, budget="standard")

piles = [f for f in report.findings if f["type"] == "boundary_mass"]
print(f"{len(piles)} boundary_mass findings")
for f in piles:
    print("-", f["claim"])

Three of the ten statements are piles at a maximum.
`capital_gain` piles 244 rows at 99,999, against a local baseline
of 7 — the median count among the ten distinct values just below
the cap, the nearest of which, 41,310, occurs 3 times — for a
dominance ratio of 34.9. `hours_per_week` piles 137 rows at 99
(45.7×) and `age` piles 55 rows at 90 (6.5×). Both caps are
documented for this extract: 99,999 is the Census Bureau's
capital-gain cap, and 90 its top-code for age. The claims state
none of that. Each states the pile, its count and share, and the
baseline it dominates; the cap reading sits in the finding's
interpretation, named as one possible cause.

The other seven statements are interior mass points in the two
capital columns, which are zero on 91.7% and 95.3% of rows.
`capital_gain` heaps at 15,024 (513 rows), 7,688 (410), and 7,298
(364); `capital_loss` at 1,902 (304), 1,977 (253), 1,887 (233),
and 2,415 (72). Each is measured against the median counts of its
nearest distinct values on each side, and reported against the
larger of the two.

`capital_loss` also holds the table's largest repeated value —
46,560 rows at 0 — and the check makes no statement about it. A
value holding more than a quarter of the evaluated rows is the
column's floor, not a pile at its edge, and a column that is zero
on 95.3% of its rows is ordinary zero inflation. The share cap is
what keeps the statements above to piles that stand out from a
bulk, rather than restating the bulk.

In [ ]:
import altair as alt

tail = (
    df["capital_gain"].value_counts().sort("capital_gain")
    .filter(pl.col("capital_gain") >= 10_000)
    .rename({"capital_gain": "value", "count": "rows"})
)
(alt.Chart(tail).mark_bar()
 .encode(
     x=alt.X("value:O", sort=tail["value"].to_list(), title="capital_gain"),
     y=alt.Y("rows:Q", title="Rows"),
 )
 .properties(
     title="Rows per distinct capital_gain value, values of 10,000 and up",
     width=620, height=300))

What to look for: the counts thin out as values rise — the
largest value below the cap, 41,310, carries 3 rows — and then a
single bar of 244 rows stands at 99,999, with no values between.
The taller bar at 15,024 (513 rows) is the interior heap: it
stands among neighbours carrying tens of rows, and its statement
measures it against the median count on each side of it — 25 on
the low side and 7 on the high side. The pile at the edge has
neighbours on one side only, and is measured against the values
below it alone.

In [ ]:
window = (
    df["capital_loss"].value_counts().sort("capital_loss")
    .filter((pl.col("capital_loss") >= 1_500) & (pl.col("capital_loss") <= 2_800))
    .rename({"capital_loss": "value", "count": "rows"})
)
(alt.Chart(window).mark_bar()
 .encode(
     x=alt.X("value:O", sort=window["value"].to_list(), title="capital_loss"),
     y=alt.Y("rows:Q", title="Rows"),
 )
 .properties(
     title="Rows per distinct capital_loss value, 1,500 to 2,800",
     width=620, height=300))

What to look for: most values in this window carry between 1
and 67 rows. Three bars — 1,887, 1,902, and 1,977, at 233, 304,
and 253 rows — stand clear of that range. The fourth reported
point, 2,415 at 72 rows, does not stand out against the window as
a whole; it stands out against its own nearest values, whose
median counts are 7.5 on the low side and 4 on the high side.
Both charts make the same point: the comparison is against the
values next to the pile, never against the column as a whole.

## Eligibility

The check scans numeric columns — integer and float. Text and
categorical columns are never candidates. A column is examined
when it holds at least 100 evaluated rows and at least 20
distinct values. Evaluated rows are the rows where the column is
present: nulls are not evaluated, and the evidence counts them
separately; non-finite float values are not values.

A column under either floor produces no statement, and the
report does not name it. The `boundaries` coverage entry counts
the table's numeric columns; it does not distinguish the columns
the floors passed over. A missing finding on a column therefore
reads two ways — the column was examined and no value cleared
the rule, or the column was never eligible — and only the
column's own row and distinct counts separate the two readings.
An ineligible column is not a cleared column.

At most four findings are reported per column: boundary piles
first, then interior mass points by count.

In [ ]:
small = pl.DataFrame({"capped": [min(10.0 + (i % 37), 42.0) for i in range(90)]})
narrow = pl.DataFrame({"capped": [float(min(i % 12, 11)) for i in range(5_000)]})

for name, frame in [("small", small), ("narrow", narrow)]:
    r = onceover.profile(frame)
    found = [f for f in r.findings if f["type"] == "boundary_mass"]
    at_max = frame.filter(pl.col("capped") == frame["capped"].max()).height
    print(f"{name}: {frame.height} rows, {frame['capped'].n_unique()} distinct values, "
          f"{at_max} rows at the maximum -> {len(found)} boundary_mass findings; "
          f"coverage: {r.data['coverage']['boundaries']['detail']}")

The small frame piles 10 of its 90 rows at its cap of 42, and
the narrow frame piles 416 of its 5,000 rows at its maximum of
11. Neither earns a statement, and the coverage line has the same
form as the one the Adult run produced for columns that were
examined. The floors are doing their work — at 90 rows a repeated
cap cannot be separated from a cluster of genuine values, and at
12 distinct values a column is a count, not a continuum with
edges — but the silence itself does not say which case a column
is in.

## The rule

One exact value-count scan per eligible column produces the
distinct values and their counts. Three kinds of candidate are
tested against the counts of their nearest distinct values.

**A pile at the maximum or minimum** is reported when the value
occurs at least 10 times, holds at most 25% of the evaluated
rows, leaves at least 40 other rows, and occurs at least 4 times
the median count of the up-to-10 distinct values just inside the
support. The baseline is local because a global comparison fails
on this table: `capital_gain`'s interior heap at 7,688 holds 410
rows, more than the cap's 244, so any bar set in raw counts ranks
the heap above the pile. What separates the cap is that its
neighbours are nearly empty.

**A pile at exactly zero** must clear the same bars and one more:
the gap to the next distinct value must be at least the
interquartile range of the bulk — the column with the pile
removed. A zero that touches its bulk is a floor mass, the
ordinary shape of a zero-inflated column, not an isolated pile.

**An interior mass point** is tested only in a zero-inflated
column — at least half the evaluated rows are zero — and only
among its nonzero values. The count floor is higher, 20
occurrences against the boundary's 10: an interior claim has no
support edge to stand on. The value must occur at least 4 times
the median count on each side of it separately, up to 10 distinct
values a side, and the reported ratio uses the larger of the two
side medians. A one-sided test fires all along a monotone
decline, where every low value beats the median of its thinner
upper neighbours; a genuine mass point stands out from both
directions. The smallest and largest nonzero values are not
interior candidates: a pile at the edge of the nonzero support
is a question about the recording grid, not about an interior
amount.

One exemption covers both boundary kinds. A column whose
observed support is exactly a canonical bounded scale — (0, 1),
(0, 5), (0, 10), (0, 100), (1, 5), (1, 7), or (1, 10) — never
reports its own bounds: mass there is the scale working as
designed.

In [ ]:
import random

rng = random.Random(11)
raw = [rng.gauss(20.0, 5.0) for _ in range(5_000)]
planted = sum(1 for v in raw if v > 30.0)
capped = pl.DataFrame({"reading": [min(v, 30.0) for v in raw]})

cap_report = onceover.profile(capped)
for f in cap_report.findings:
    if f["type"] == "boundary_mass":
        print(f["claim"])
print(f"values moved to the cap when the frame was built: {planted}")

The scan recovers the planted count exactly: 124 values were
moved to the cap when the frame was built, and 124 is the count
the claim states. The baseline is 1 because the distinct values
of a continuous column occur once each, so the dominance ratio
is the count itself. What the planted case cannot show is
discrimination — a continuous column's neighbours are all
singletons — which is what the next frame is for.

In [ ]:
import math

vals = []
for score in range(5, 101):
    count = round(300 * math.exp(-(((score - 72) / 18.0) ** 2))) + 20
    vals.extend([float(score)] * count)
scores = pl.DataFrame({"score": vals})

score_report = onceover.profile(scores)
found = [f for f in score_report.findings if f["type"] == "boundary_mass"]
top = scores["score"].value_counts().sort("score").tail(3)
print(f"{scores.height} rows; boundary_mass findings: {len(found)}")
print("counts at the top of the support:",
      [(int(v), c) for v, c in zip(top["score"].to_list(), top["count"].to_list())])

The scores frame piles 47 rows at its maximum of 100 — nearly
five times the count floor — and earns no statement, because its
neighbours are as crowded as the edge: 52 rows at 99 and 57 at
98, so the dominance ratio is below 1, let alone 4. Its support,
5 to 100, is not one of the canonical scales, so the exemption is
not what keeps it quiet. A pile is a statement about local
dominance, not about how many rows sit at the edge.

In [ ]:
metro = pl.read_csv(
    "../../../benchmarks/corpus/data/anomaly/metro-interstate-traffic/data.csv"
)
metro_report = onceover.profile(metro, budget="standard")
for f in metro_report.findings:
    if f["type"] == "boundary_mass":
        print("-", f["claim"])
        if f["evidence"][0]["metrics"]["boundary"] == "minimum":
            m = f["evidence"][0]["metrics"]
            print("   temp evidence:", {k: m[k] for k in
                  ("gap_to_neighbour", "bulk_iqr", "dominance_ratio")})

In [ ]:
low = (
    metro["temp"].value_counts().sort("temp").head(12)
    .rename({"temp": "value", "count": "rows"})
)
(alt.Chart(low).mark_bar()
 .encode(
     x=alt.X("value:O", sort=low["value"].to_list(), title="temp"),
     y=alt.Y("rows:Q", title="Rows"),
 )
 .properties(
     title="Rows per distinct temp value, the 12 smallest values",
     width=620, height=300))

What to look for: the bar at 0.0 is the only value in the
window carrying more than 3 rows, and the next value up is
243.39. The gap the claim states — 243.39 from the pile to the
nearest reading — is more than twelve times the bulk's
interquartile range of 19.63.

`temp` is the floor case, and the zero-isolation rule is what
carries it. The ten readings of exactly 0.0 clear the count floor
and the dominance bar — the median count of the ten nearest
higher values is 1 — and the gap clears the isolation bar of one
bulk interquartile range. Whether 0.0 can occur for this quantity
at all — the column is recorded in kelvin — depends on units and
domain knowledge the profile does not have, so the statement
rests on the distribution alone: the count, the gap, and the
baseline. The same run reports four interior mass points in
`rain_1h`, at the rain gauge's recording ticks: 0.51 on 256 rows
(39.4× the larger side baseline), 1.02 on 123 rows (20.5×), 0.3
on 121 rows (18.6×), and 0.76 on 109 rows (21.8×).

## Reading the evidence

Every finding's evidence carries the quantities the claim was
built from, so the statement can be checked without re-running
the scan. `value` and `count` give the pile; `share` is its
share of the `n` evaluated rows, read against `population_rows`
(the table's full row count) and `null_rows` (the rows not
evaluated). `boundary` names the kind: `maximum`, `minimum`, or
`interior`. `distinct_values` counts the column's support.

The comparison itself is in `neighbour_value` and
`neighbour_count` — the nearest distinct value inside the
support, and its count — in `neighbour_baseline`, the median
count over the `neighbour_window` nearest distinct values, and
in `dominance_ratio`, the pile's count divided by that baseline.
An interior finding carries `below_baseline` and `above_baseline`
instead, and its ratio uses the larger of the two. The rest of
the column is described with the pile excluded: `bulk_min`,
`bulk_max`, `bulk_median`, and `bulk_iqr` over
`bulk_distinct_values` distinct values, with `gap_to_neighbour`
and the column's `zero_count` and `zero_share`. The implicated
rows are named by `example_rows`, the first ten row indices
holding the value, with `first_row` and `last_row`. The `method`
field restates the rule in one string.

## What the check does not establish

A pile is a statement about the data as given. Read as a
recording artifact, it says the piled rows were recorded as the
boundary value — everything at or beyond a cap written as the
cap — so the recorded value is a bound on the true values, not
the values themselves: at a maximum cap, each piled row's true
value is at least the recorded one. Read as a property of the
quantity, the same shape is a genuine bound saturating: a
percentage that cannot exceed 100 piles there with nothing
censored. The profile cannot distinguish the two readings, and
the finding's interpretation says so in those terms — a
recording cap or floor is named as one possible cause, never as
the conclusion.

A pile is not, by itself, evidence that values were manipulated
or mishandled. Caps and floors are ordinary recording practice:
top-coded income and floored sensors are published that way
deliberately. The check's contribution is narrower. It states
that the edge value is crowded far beyond its neighbours, with
the counts attached, so a reader computing a mean, a spread, or
a model on the column knows the edge is a pile and not a tail.

## What the check does not report

- **A value holding more than a quarter of the rows.** It is the
  column's mode or floor, however sharp the edge looks:
  `capital_loss`'s 46,560 zeros are the measured case.
- **A smooth distribution reaching its bound.** Where the values
  just inside the edge carry comparable counts — graded scores
  crowding the top of their range — the dominance test fails, as
  the scores frame above shows.
- **The bounds of a canonical scale.** A column spanning exactly
  0–100, 1–5, or any of the other listed pairs never reports its
  own bounds.
- **A zero pile that touches its bulk.** If the next distinct
  value sits closer than one bulk interquartile range, the zeros
  are read as ordinary zero inflation.
- **Interior points outside zero-inflated columns.** A repeated
  amount in a column that is not at least half zeros is left to
  the column card's top values; the interior test's context is
  the zero-inflated column, where an exact repeated amount is a
  recording fingerprint rather than a mode.
- **A repeated value concentrated in a stretch of rows.** That
  is the sentinel test's shape, covered on [its own
  page](sentinel_values.ipynb). This check counts over the whole
  column; where the piled rows sit plays no part in it.
- **Anything past the fourth finding in a column.** Boundary
  piles are reported first, then interior points by count; the
  cap keeps one crowded column from filling the report.
- **Columns under the eligibility floors.** Ninety rows or
  twelve distinct values earn silence, not clearance — see
  Eligibility above.